# 37 — LLM Systems Foundations

### Why this exists
An LLM application is a system around a model: context, instructions, generation, validation, policy and evaluation. This lab creates a deterministic fake model so the engineering boundaries remain inspectable offline.

### Learning objectives
trace an AI request; separate prompt, context and output; identify failure boundaries

### Prerequisite
Python dictionaries and functions

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/37-llm-systems-foundations.svg)

> **Question to carry through the lab:** Where can a correct model still produce an incorrect product outcome?

## Capability contract

**Capability:** C08 — LLM Systems  
**Workstream:** AI Support Platform  
**Primary roles:** AI Engineer|ML Engineer  
**You will prove:** Build or inspect a grounded AI mechanism


## Simulation → reality bridge

**Real-system bridge:** Provider-neutral LLM runtime

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook uses deterministic model behavior to isolate prompts, context and output contracts. Real models add stochasticity, context limits, latency, token cost, model/version drift and provider-specific failure modes.

**Transfer challenge:** Replace the deterministic model with any compatible provider adapter available to you and compare which guarantees remain true.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


## Mental model
Model capability is only one component. System behavior emerges from input construction, context quality, output constraints, post-validation and downstream action policy.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S16 — 1181: LLM response latency is unpredictable** → [`../work_simulations/16_acme_commerce_llm_contract/README.md`](../work_simulations/16_acme_commerce_llm_contract/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
def fake_model(prompt,context):
    if "refund" in prompt.lower() and "refund_policy" in context: return {"answer":"Eligible under policy", "source":"refund_policy"}
    return {"answer":"I do not have enough grounded context", "source":None}
request={"prompt":"Can I get a refund?","context":{"refund_policy":"refund within 30 days"}}
print(fake_model(**request))

In [ ]:
def trace(req):
    output=fake_model(req["prompt"],req["context"])
    return {"request":req,"model_output":output,"validated":bool(output.get("source"))}
print(trace(request))

In [ ]:
bad={"prompt":"Can I get a refund?","context":{}}
print(trace(bad))

In [ ]:
# Break downstream policy: an ungrounded answer must not become an action.
def authorize(trace_record):
    return trace_record["validated"] and trace_record["model_output"].get("source")
print("action allowed",authorize(trace(bad)))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production AI systems need input/version tracing, model/version metadata, policy enforcement, evaluation, cost/latency budgets and safe downstream boundaries.

## 9. Independent challenge
Replace the fake model with two deterministic failure cases: stale context and a conflicting policy. Decide which component should own each failure.

### Evidence to keep
end-to-end trace, groundedness flag, ungrounded failure, and action authorization decision

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.